# Build Dataset502_ARCADE_6x6_1c

Builds an nnUNet-style raw dataset from the ARCADE `syntax` subset:
grayscale images, binary vessel mask (all vessel-segment classes collapsed to foreground),
tiled into a 6x6 layer-1 grid plus 5x5 border-straddling layer-2 patches.

Output: `data/nnUNet_raw/Dataset502_ARCADE_6x6_1c/`

In [ ]:
import sys
from pathlib import Path

import json
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

sys.path.insert(0, str(Path.cwd()))

import prepare_arcade as pa
from patch_grid import layer1_boxes, layer2_boxes

In [ ]:
out_root, train_cases, val_cases, test_cases = pa.build_dataset(
    dataset_id=502,
    source="syntax",
    cols=6,
    rows=6,
    include_layer2=True,
)

print(f"dataset: {out_root}")
print(f"train patches: {len(train_cases)}")
print(f"val patches:   {len(val_cases)}")
print(f"test patches:  {len(test_cases)}")

In [ ]:
print((out_root / "dataset.json").read_text())
splits = json.loads((out_root / "splits_final.json").read_text())
print(f"splits_final.json: fold0 train={len(splits[0]['train'])} val={len(splits[0]['val'])}")

## Grid overlay preview

Layer-1 boundaries (solid) and layer-2 patch outlines (dashed), on one source image.

In [ ]:
source_dir = pa.ARCADE_ROOT / "syntax"
sample_img_path = source_dir / "train" / "images" / "1.png"

with Image.open(sample_img_path) as im:
    gray = np.array(pa.to_grayscale(im))

width, height = im.size
l1 = layer1_boxes(width, height, 6, 6)
l2 = layer2_boxes(width, height, 6, 6)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(gray, cmap="gray")

for row in l1:
    for (x0, y0, x1, y1) in row:
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="lime", linewidth=1))

for row in l2:
    for (x0, y0, x1, y1) in row:
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="red", linestyle="--", linewidth=1))

ax.set_title("6x6 layer-1 (green) + 5x5 layer-2 (red, dashed)")
ax.axis("off")
plt.show()

## Sample patches

A few layer-1 and layer-2 image/mask pairs straight from the written dataset.

In [ ]:
def show_cases(case_ids, n=4, title=""):
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    for i, case_id in enumerate(case_ids[:n]):
        img = Image.open(out_root / "imagesTr" / f"{case_id}_0000.png")
        mask = Image.open(out_root / "labelsTr" / f"{case_id}.png")
        axes[0, i].imshow(img, cmap="gray")
        axes[0, i].set_title(case_id, fontsize=8)
        axes[0, i].axis("off")
        axes[1, i].imshow(np.array(mask), cmap="gray", vmin=0, vmax=1)
        axes[1, i].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

layer1_sample = [c for c in train_cases if "_p" in c and c.startswith("train_1_")]
layer2_sample = [c for c in train_cases if "_b" in c and c.startswith("train_1_")]

show_cases(layer1_sample, n=4, title="layer-1 patches (train_1)")
show_cases(layer2_sample, n=4, title="layer-2 border patches (train_1)")